# Reduction and Parallel Streams

In this lesson, you will learn to combine values with reductions and explain the rules for correct sequential and parallel results.

CSC-239 · Module 10 · Lesson 4 of 4

You have used lambdas to describe behavior and fresh streams to process collections. Now you will make one total from several values. You will also explain why a valid combining rule works when partial totals are calculated separately.

Use the [module glossary](terms.md) to revisit unfamiliar terms after reading their explanations here.


## Learning Goals

- Build and test sequential and parallel reductions with a neutral identity and an associative combining rule.
- Explain the results for empty input, rejected entries, repeated entries, and one retained value.
- Distinguish correct results from claims about execution order or speed, while keeping processing rules independent and the source unchanged.


## Why This Matters

An inventory report, attendance summary, or sales dashboard often needs one total rather than another list. A reduction expresses how individual contributions become that result. You can check the combining rule separately from the rest of the report, including what happens when there are no contributions.

Larger applications may divide a calculation into parts. The rule must remain correct when those partial results are combined. Understanding identity, grouping, and shared state helps you decide whether that change is valid. Requesting parallel work alone does not establish correctness or make a program faster; the actual workload still needs measurement.


## Check Your Starting Point

Recall an accumulator loop that adds selected values into a total. Explain what its starting value does and how one selected value changes the total. Then explain why two terminal stream computations need two fresh streams. Finally, describe what happens when two references update the same mutable object.

Use examples from the earlier lessons. These distinctions will help you compare a local accumulation with a combination of independent partial results.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

An addition loop commonly starts its total at zero and replaces the total with the old total plus each selected contribution. A terminal operation consumes its stream, so another computation must obtain a fresh stream from the collection. The source collection can remain available after a stream finishes.

Two references to one mutable object reach the same stored state. A change through one reference can be observed through the other. An unchanged reference does not make its object immutable. Keep that distinction in mind when a processing rule would update a shared total.

</details>


## Video Demonstration

Watch a supply count receive separate sequential and parallel reductions. The example connects zero and addition to the rules that make both totals valid.

<video controls preload="metadata" width="960">
  <source src="media/04_reduction_and_parallel_streams/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_reduction_and_parallel_streams/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the reduction and parallel streams demonstration transcript](media/04_reduction_and_parallel_streams/transcript.md).


## Concept

### Combine values into one result

A campus supply clerk records the quantities in two deliveries: three notebooks and five notebooks. Each list entry counts individual notebooks in one delivery. The clerk needs the total number received, counting every delivery once. No entries are filtered or changed in this first example.

A **reduction** combines several values into one result. The `reduce` operation below is a terminal operation: it performs the requested processing and returns the total. Its first argument is a starting value. Its second argument supplies the rule for combining two values.

The rule `(left, right) -> left + right` has two parameter names in parentheses. It receives two values and returns their sum. In a sequential trace, the first input can be the total so far and the second the next quantity. Later, when partial results are combined, both inputs can be partial totals. The names describe inputs to the rule; they are not shared variables that the lambda updates.

This form of `reduce` expects a **BinaryOperator**, a functional interface for two inputs and a result of the same type. Here the collection's element type is `Integer`, so the combining rule works with `Integer` values. The familiar unboxing and boxing conversions allow the addition expression to use integer arithmetic. A separately stored `BinaryOperator<Integer>` can also be called through its `apply` method with two arguments.

In the complete program, the import makes `ArrayList` available, the constructor creates the list, and the two `add` calls record the deliveries. `stream()` supplies a fresh description of the list's entries. The terminal `reduce` returns one value, which is assigned to `total` before the print statement runs.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(3);
counts.add(5);
int total = counts.stream().reduce(0, (left, right) -> left + right);
System.out.println("Total: " + total);


The output is:

```text
Total: 8
```

One sequential trace starts at zero, combines zero with three to obtain three, then combines three with five to obtain eight. The result is eight individual notebooks. The lambda returns each combined value; it does not change an extra shared total or edit the list.

That trace explains this input, but a reusable rule must also explain what its starting value means when the list is empty.


### Choose a neutral value, including for no entries

An **identity value** is neutral under the combining rule: combining it with a valid value leaves that value unchanged. Zero is the identity for addition because both `0 + value` and `value + 0` equal `value`. It contributes no extra inventory.

An empty delivery list should therefore total zero. The next program creates the list without adding entries. The terminal reduction has no quantities to combine, so it returns its identity argument directly. The list is empty; the returned total is still a useful result.

Starting at one would count an item that no delivery supplied. It would also fail to be neutral when partial results are combined. Choose an identity that works for the operation, not a starting value that happens to produce a desired answer for one list.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> empty = new ArrayList<Integer>();
int total = empty.stream().reduce(0, (left, right) -> left + right);
System.out.println("Empty total: " + total);


The output is:

```text
Empty total: 0
```

No combining call is needed to account for source entries because there are none. Zero is the result of the empty reduction. An empty-input test checks this part of the contract; it does not replace checking whether the combination rule permits regrouping.


### Check regrouping before requesting parallel work

An **associative combination** preserves its result when the same ordered values are grouped differently. With two, four, and one item, addition can combine the first pair or the last pair first:

```text
(2 + 4) + 1 = 7
2 + (4 + 1) = 7
```

The parentheses select which partial calculation happens first. They do not change the quantities or their order. Both groupings give seven, which is why independently computed partial sums can be combined into the same total.

Subtraction exposes a different rule:

```text
(8 - 3) - 1 = 4
8 - (3 - 1) = 6
```

The same ordered inputs now give different results. That counterexample is enough to show that subtraction is not associative. It cannot satisfy the combining contract of this reduction, even if a particular attempted parallel run happens to give an expected number.

Identity and associativity answer separate questions. Identity asks whether an extra neutral value changes a result. Associativity asks whether regrouping changes a result. Both requirements matter. The exercises use small integers with totals that fit in `int`; do not assume these examples establish the behavior of every numeric type.


### Combine possible partial results

**Parallel stream processing** allows Java to divide pipeline work into parts that may run at the same time and then combine partial results. Calling `parallelStream()` requests this mode for the collection. It does not promise a particular number of workers, division of values, or execution schedule.

For the clerk's three-item and five-item deliveries, a possible grouping would calculate `0 + 3` in one part and `0 + 5` in another, then add those partial totals. Each part uses the neutral zero without inventing an item. Associative addition allows the partial sums to be combined correctly. This is a possible grouping for understanding the contract, not a report of what Java actually scheduled.

The complete program below creates separate fresh streams. The call using `stream()` requests the sequential calculation; the call using `parallelStream()` requests the parallel calculation. Both supply the same identity and combining rule. Each terminal operation returns before its assignment completes.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(3);
counts.add(5);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


The complete output is:

```text
Sequential: 8
Parallel: 8
```

Both computations account for the same three and five items. The caller prints the sequential total first because the two `println` statements execute in their written order after the reductions. Those two output lines reveal the complete results. They do not reveal which source entry a parallel worker handled first.

Parallel processing also requires coordination and combination work. A small list may take longer to process in parallel. Matching totals support correctness for this run; they do not establish a speed improvement. We will judge these exercises by the operation's rules and outputs, not elapsed time.


### Keep each operation independent and the source stable

Correct arithmetic is only part of the contract. **Stateless behavior** means a processing rule does not rely on changing state shared between its calls. For example, `value -> value + 1` calculates its result from the supplied value. If the input is four, the returned value is five regardless of which other entry was processed first.

A mapping that reads and updates a shared running total would have a different meaning: its result could depend on earlier calls. Separate pieces of parallel work could also try to read and replace that total at the same time. Keep the mapping's calculation inside its returned expression, and let `reduce` combine returned values. Do not place the shared total inside an array or object merely to bypass a captured-local-variable restriction; the object would still be shared and mutable.

**Noninterference** means leaving the source unchanged while its pipeline executes. A rule that adds to or removes from `counts` would interfere with the entries being processed. This differs from whether a rule depends on some other shared counter. A rule can violate either requirement independently.

For a related example, applying the add-one mapping to four, one, and two returns five, two, and three. Summing those returned values gives ten. The original list still contains four, one, and two because no stage assigns new list entries. You will check this distinction directly in practice. Creating new values through a mapping does not require changing their source collection.

We can now apply the identity, grouping, and state requirements together in a complete item-count report.


## Worked Example

### Total three supply deliveries

The supply clerk now records three deliveries containing two, four, and one item. Each entry is an item quantity, and every entry contributes once. The report must show a sequential total and a parallel total for exactly the same list. Their units are items; this example does not measure processing time.

**Choose the data and combining rule.** Create `ArrayList<Integer> counts` and add the three quantities. Use zero as the identity because an empty set of deliveries contributes no items. Addition combines either a quantity with an accumulated amount or two partial totals without adding unrelated state.

**Request separate computations.** In `counts.stream().reduce(0, (left, right) -> left + right)`, the source call creates a fresh sequential stream and `reduce` returns the total stored in `sequential`. The next expression starts again from `counts` through `parallelStream()` and stores its result in `parallel`. It does not try to reuse the consumed first stream.

**Report complete results.** The two print statements use those returned values. Neither combining lambda changes a shared variable or edits `counts`. The complete program below puts the setup, computations, and report together.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(2);
counts.add(4);
counts.add(1);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


Expected output:

```text
Sequential: 7
Parallel: 7
```

The sequential total includes two, four, and one item, giving seven. For a possible parallel grouping, one partial sum could be six from the first two entries and another one from the last entry; combining them also gives seven. Zero changes neither partial sum. This explains why the result is valid without claiming that this particular grouping occurred.

The caller prints the two complete totals in the order shown. Matching values check this report, while the identity and associativity reasoning explain why permitted regrouping preserves it. The calculation supplies no evidence about which worker handled each entry or which execution mode was faster.


<details id="animation-parallel-partial-reduction" class="animation-panel" open>
<summary>Trace the example — show or hide animation</summary>
<p><img src="media/04_reduction_and_parallel_streams/parallel-partial-reduction.gif" alt="Quantities two, four, and one add to seven. A possible parallel grouping combines partial sums six and one to reach the same total." width="960" style="max-width:100%;height:auto;"></p>
</details>

Trace the sequential total, then compare one possible grouping of partial sums. The printed results establish the totals; they do not reveal how Java divided the work. This silent loop lasts 22 seconds. Hide the animation to remove its visible motion. [View the final state as a still image](media/04_reduction_and_parallel_streams/parallel-partial-reduction_still.png).

## Guided Practice

Use the next tasks to separate three decisions: the values being combined, the rule that combines them, and the conclusions the output supports. Predict before running, then keep your prediction when you explain the observed result.


### Predict two reductions

The next program uses the quantities five, zero, and six. Predict both printed lines before running. Explain the separate roles of the zero source entry and the zero identity, identify the combining lambda's two inputs and result, and justify whether the sequential and parallel totals should agree.


In [ ]:
Your response:


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


Record the actual output and explain any correction to your prediction. Identify the fresh source and terminal operation for each computation.

Trace a sequential total from the identity through all three entries. Give two different groupings of five, zero, and six that preserve their order, and distinguish that grouping check from the identity check. Explain whether the rules change the source or a shared total. State what the two print statements establish about the results, and what they cannot establish about parallel element order or speed.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

Both reductions combine five, zero and six to produce eleven. The identity zero contributes no extra amount, including when it is combined with a partial result. Addition of these small integers gives the same total under regrouping. Each reduction receives a fresh stream: stream requests sequential processing, while parallelStream requests parallel processing. The two-parameter lambda returns the sum of its arguments without updating a shared total or changing counts. The caller prints Sequential: 11 and then Parallel: 11 after the corresponding results have been computed. These print statements do not reveal the processing order of individual elements. A sequential trace can start at zero and combine to obtain five, then five, then eleven. For example, grouping five with zero first or zero with six first still produces eleven. This is a grouping check, not a claim about how the parallel implementation divided this run. The BinaryOperator pattern matches two Integer inputs and an Integer result; the familiar boxing and unboxing rules support the arithmetic. Matching totals confirm these results, but do not prove a speed improvement.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Adding the identity as an extra nonzero contribution. Assuming parallel processing means a different mathematical operation. Treating equal totals as proof of a particular element-processing order.

</details>


### Check neutrality and regrouping directly

The next program stores addition in a `BinaryOperator<Integer>` named `plus`. Its `apply` method receives two `Integer` values and returns their combined `Integer` result. Predict all six output lines before running. Identify which calls check the identity and which check grouping, then compare the two subtraction expressions.


In [ ]:
Your response:


In [ ]:
import java.util.function.BinaryOperator;
BinaryOperator<Integer> plus = (left, right) -> left + right;
System.out.println("Left identity: " + plus.apply(0, 6));
System.out.println("Right identity: " + plus.apply(6, 0));
System.out.println("Left grouping: " + plus.apply(plus.apply(6, 2), 3));
System.out.println("Right grouping: " + plus.apply(6, plus.apply(2, 3)));
System.out.println("Left subtraction: " + ((9 - 4) - 2));
System.out.println("Right subtraction: " + (9 - (4 - 2)));


Record all six actual lines and explain any corrections. Explain why testing a neutral identity is different from testing regrouping. Use the subtraction results to explain why subtraction cannot satisfy this reduction's associative combining rule. These are direct arithmetic checks, not observations of an invalid parallel schedule.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

The first two calls return six because zero is neutral on either side of addition. The next calls combine six, two and three with different groupings and both return eleven. The plus value has type `BinaryOperator<Integer>`: its apply call accepts two Integer inputs and returns an Integer result. The subtraction expressions return three and seven because regrouping subtraction changes its result. That counterexample shows why subtraction cannot satisfy the associative rule required here. These are direct arithmetic checks; the program does not run an invalid parallel reduction or claim a fixed result for one.

```java
import java.util.function.BinaryOperator;
BinaryOperator<Integer> plus = (left, right) -> left + right;
System.out.println("Left identity: " + plus.apply(0, 6));
System.out.println("Right identity: " + plus.apply(6, 0));
System.out.println("Left grouping: " + plus.apply(plus.apply(6, 2), 3));
System.out.println("Right grouping: " + plus.apply(6, plus.apply(2, 3)));
System.out.println("Left subtraction: " + ((9 - 4) - 2));
System.out.println("Right subtraction: " + (9 - (4 - 2)));
```

Expected output:

```text
Left identity: 6
Right identity: 6
Left grouping: 11
Right grouping: 11
Left subtraction: 3
Right subtraction: 7
```

Common error: Treating a neutral identity as sufficient without checking grouping. Changing the order of values when asked only to regroup them. Memorizing one result of an invalid parallel operation instead of checking its contract.

<details id="animation-identity-and-associative-grouping" class="animation-panel" open>
<summary>Trace the example — show or hide animation</summary>
<p><img src="media/04_reduction_and_parallel_streams/identity-and-associative-grouping.gif" alt="Zero leaves six unchanged on either side of addition. Two addition groupings return eleven, while the subtraction groupings return three and seven." width="960" style="max-width:100%;height:auto;"></p>
</details>

Compare the identity checks with the grouping checks. A neutral identity and an associative rule are separate requirements. This silent loop lasts 19 seconds. Hide the animation to remove its visible motion. [View the final state as a still image](media/04_reduction_and_parallel_streams/identity-and-associative-grouping_still.png).

</details>


### Check independent rules and the unchanged source

The next program starts with counts four, one, and two. Each mapping returns its input plus one before reduction. Predict both totals and every source line. Trace the mapping result for each original entry and identify whether any stage assigns a new value into the source list.


In [ ]:
Your response:


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(4);
counts.add(1);
counts.add(2);
int sequential = counts.stream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
for (int value : counts) {
    System.out.println("Source: " + value);
}


Compare all actual output lines with your prediction. Explain why the original source values remain available after both computations.

Assess two proposed edits in words: making the lambda read and update one shared changing total, and making it add an entry to `counts` while the pipeline runs. Identify the statelessness or noninterference requirement each would violate. Explain why the printed report does not show the parallel processing order or establish a speed improvement.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

Mapping produces five, two and three, which sum to ten in either computation. The lambda derives each mapped value from its own input, and the combining lambda derives its result from its two arguments. Neither updates a shared total. The source remains four, one and two, as its later printed entries show. A proposed rule that depends on a changing shared total violates stateless behavior. A proposed addition to counts from a running pipeline operation violates noninterference. The caller prints only after computing the totals and then reads the source list; those lines do not show which parallel element was processed first. Matching totals also provide no evidence that parallel processing was faster.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(4);
counts.add(1);
counts.add(2);
int sequential = counts.stream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
for (int value : counts) {
    System.out.println("Source: " + value);
}
```

Expected output:

```text
Sequential: 10
Parallel: 10
Source: 4
Source: 1
Source: 2
```

Common error: Treating a copied reference to a shared mutable object as independent state. Adding to the source from a running processing rule. Using the order of later caller prints as evidence of element scheduling.

<details id="animation-stateless-map-and-source-preservation" class="animation-panel" open>
<summary>Trace the example — show or hide animation</summary>
<p><img src="media/04_reduction_and_parallel_streams/stateless-map-and-source-preservation.gif" alt="Mapping four, one, and two to five, two, and three gives ten. The source still contains four, one, and two after both computations." width="960" style="max-width:100%;height:auto;"></p>
</details>

Follow each returned mapping value into the total. Computing a new value does not replace the corresponding source entry. This silent loop lasts 19 seconds. Hide the animation to remove its visible motion. [View the final state as a still image](media/04_reduction_and_parallel_streams/stateless-map-and-source-preservation_still.png).

</details>


### Complete the paired reductions

Replace SEQUENTIAL_SOURCE, PARALLEL_SOURCE, IDENTITY and COMBINE. Use stream and parallelStream for the two fresh sources, and choose the neutral value and arithmetic operation for addition. Replace each repeated placeholder consistently. Copy the whole completed program into the work cell and run it. Explain why both computations need the same valid identity and combining operation.

This sample is for repair:

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.SEQUENTIAL_SOURCE().reduce(IDENTITY, (left, right) -> left COMBINE right);
int parallel = counts.PARALLEL_SOURCE().reduce(IDENTITY, (left, right) -> left COMBINE right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```


In [ ]:
Your response:


Run your completed program and record both actual lines. Explain how each replacement fits its purpose: obtaining a fresh stream, supplying a neutral identity, and returning a combined value. Compare the two totals without drawing a conclusion about speed.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

Use stream, parallelStream, zero and the plus operator for the four placeholders. Each source method creates a fresh computation, and each reduce uses the addition contract. Repeating the same valid identity and operation lets independently combined parts produce the same result. Both reductions combine five, zero and six to produce eleven. The identity zero contributes no extra amount, including when it is combined with a partial result. Addition of these small integers gives the same total under regrouping. Each reduction receives a fresh stream: stream requests sequential processing, while parallelStream requests parallel processing. The two-parameter lambda returns the sum of its arguments without updating a shared total or changing counts. The caller prints Sequential: 11 and then Parallel: 11 after the corresponding results have been computed. These print statements do not reveal the processing order of individual elements.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Using a nonneutral starting value to force one desired result. Changing the arithmetic in only one pipeline. Leaving a placeholder in the executable program.

</details>


### Transform each count before reducing

The work cell starts with five, zero, and six and the existing pair of reductions. Add a `map` operation before `reduce` in both pipelines so each source count is multiplied by three. Before editing, predict the transformed values and both totals. Explain why this mapping needs no shared changing total. Then edit and run the complete program.


In [ ]:
Your response:


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


Record the first actual output and compare it with your prediction. Explain how the same mapping and combining rules account for both totals.


In [ ]:
Your response:


Keep both pipelines unchanged, but replace the zero source entry with negative two. Predict the mapped values and both printed totals before making that edit. Identify whether the program has any stage that would reject a negative value. Then edit and run the same complete program.


In [ ]:
Your response:


Record the variant's output and explain why the negative contribution remains. Restore the zero source entry and rerun the complete program. Record the restored result as a check that only the intended input changed.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

The map operation changes five, zero and six into fifteen, zero and eighteen. Addition with identity zero combines them to thirty-three in both pipelines. After replacing the source zero with negative two, mapping produces fifteen, negative six and eighteen, totaling twenty-seven. This program has no filtering stage, so the negative contribution remains. Each mapping result depends only on its input; reduce combines returned values instead of requiring a shared variable.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 33
Parallel: 33
```

Common error: Adding the mapping stage to only one of the two computations. Multiplying the identity instead of each processed input. Assuming a negative input is automatically filtered out.

**Additional test: `Replace the zero source entry with negative two`.** The mapped negative input contributes negative six because there is no filter. Both complete pipelines sum fifteen, negative six and eighteen to twenty-seven with zero as the identity.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(-2);
counts.add(6);
int sequential = counts.stream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 27
Parallel: 27
```

</details>


### Repair a nonneutral starting value

The displayed sequential program runs, but it claims to add only the supplied counts. Predict its printed value and locate the extra contribution. Explain why its starting value is not an identity for addition. Repair the starting value and include a separate fresh parallel reduction using the same correct rule, with Sequential and Parallel output labels. Put your whole repaired program in the work cell and run it. Then remove all source additions and test the complete repaired pair again. Explain what the empty result establishes about the identity.

This sample is for repair:

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(1, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
```



In [ ]:
Your response:


Run your repaired sequential and parallel pair. Record both actual lines and explain which extra contribution your repair removed. Explain why the corrected value is neutral rather than merely convenient for these inputs.


In [ ]:
Your response:


Now remove all source `add` calls while keeping the repaired identity and both pipelines unchanged. Predict both printed results before running. Explain which part of the reduction contract the empty case checks. Then edit and run your complete program.


In [ ]:
Your response:


Record the empty-input results and explain whether they match the identity rule. Restore five, zero, and six as the source entries, rerun the repaired pair, and record the restored output.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

The faulty sequential calculation starts at one and then adds five, zero and six, so it prints Sequential: 12. The extra one is not a source contribution and is not neutral: one plus five is six, not five. A starting value must satisfy the identity rule, not merely fit one example. Restoring zero gives eleven for both complete repaired reductions. With all source additions removed, both repaired reductions return zero. That empty-input result is the identity itself. The faulty draft is sequential only; no fixed result for an invalid parallel reduction is asserted.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Changing the expected total to accept the extra starting contribution. Repairing one mode while leaving an invalid identity in the other. Claiming one observed invalid reduction result establishes a valid contract.

**Additional test: `Empty source with the repaired identity`.** No input values are available to combine. Both complete valid reductions return their neutral zero identity, using separate fresh streams.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

</details>


## Independent Practice

### Build the positive-score total

Create `scores` with the four `Integer` values four, negative two, zero, and seven, in that order. Keep only positive scores, double each retained score, and reduce using zero and addition. Build separate fresh sequential and parallel pipelines with the same stages. The required baseline report is `Sequential: 22` followed by `Parallel: 22` on separate lines.

Before coding, describe the input and output of each stage and justify the identity and grouping rule. Explain how your rules will avoid shared mutable accumulation and source changes. Then write the entire program, including the import, source setup, and both computations, in the Java work cell.


In [ ]:
Your response:


Run the complete baseline program and record its actual output. Identify the retained values, mapped values, and resulting total. Explain any changes needed to make your implementation match your plan.


In [ ]:
Your response:


### Check empty, rejected, single, and repeated values

Keep the filtering, mapping, identity, combining rule, and output statements unchanged. Predict the retained values, mapped values, and both totals for each separate source list before running any variant:

- An empty list.
- Negative three, zero, and negative one.
- Three, three, zero, and negative two.
- The single value five.
- The restored original list: four, negative two, zero, and seven.

Use a labeled prediction for each case. These tests concern the stage rules and results; elapsed time is not a grading condition.


In [ ]:
Your response:


Edit and run your complete program for each predicted list, then restore and run the original list. Record both actual totals for every case and explain any corrections.

Distinguish empty input from nonempty input with no retained values. Explain why both repeated positives contribute and what the single retained value checks. Use those observations and your identity/grouping reasoning to evaluate correctness without claiming a speed improvement.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

The positive-score filter retains four and seven and rejects negative two and zero. Mapping doubles the retained values to eight and fourteen. Reduction combines them to twenty-two, starting with the neutral identity zero. Both the sequential and parallel computations use fresh streams and the same stages. The addition rule uses its two supplied values instead of updating a shared mutable total. No stage edits scores. Addition of these small values preserves the total under regrouping, so both complete results print twenty-two. Empty input and input with only nonpositive scores both leave no values to reduce, so each returns zero. Their sources differ, but the reduction receives no contributions in either case. The repeated-positive input retains both threes, maps each to six and totals twelve. A single five maps to ten and reduces to ten because zero does not alter it. The same unchanged rules explain both execution modes for all four cases.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(4);
scores.add(-2);
scores.add(0);
scores.add(7);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 22
Parallel: 22
```

Common error: Doubling values without applying the required positive filter. Removing repeated matching entries even though each is a contribution. Updating a separate shared total instead of returning values for reduce.

**Additional test: Empty score list.** No source entries are present, so both fresh pipelines return the zero identity.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

**Additional test: Only nonpositive scores: -3, 0, -1.** The source has three entries, but the positive filter rejects each one. Both reductions receive no retained values and return zero.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(-3);
scores.add(0);
scores.add(-1);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

**Additional test: Repeated positives: 3, 3, 0, -2.** Both threes pass and each maps to six. Zero and negative two are rejected. The two separate contributions produce twelve.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(3);
scores.add(3);
scores.add(0);
scores.add(-2);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 12
Parallel: 12
```

**Additional test: One positive score: 5.** The single positive score maps to ten. Adding the neutral identity leaves ten unchanged in each complete pipeline.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(5);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 10
Parallel: 10
```

</details>


## Summary

A reduction combines contributions into one result. Its identity is neutral and supplies the empty result. Its combining operation must preserve the result under valid regrouping. Parallel processing can combine independent partial results when those rules hold.

Stateless behavior keeps a rule from depending on changing shared values. Noninterference keeps the source unchanged during processing. Neither matching totals nor the caller's print order establishes a worker schedule or a speed improvement.

Close the answers. Justify the identity, combining rule, source behavior, and empty-input result of your final pipeline in your own words.


In [ ]:
Your response:


<details>
<summary>Show answer</summary>

Zero is neutral for the final addition. Adding grouped partial sums preserves the total for these small integer values. Filtering and mapping return decisions and values without editing `scores` or a shared total. When no values reach reduction, zero is returned. Those facts support the calculation's correctness; assessing performance would require separate measurements of the actual workload.

</details>


## Reflection

Choose a numerical report from your field. State which values to retain, how to transform each retained value, and how to combine the results. Explain whether the combining rule is associative and what should happen when no values remain. Identify a source change or shared-state dependency your design should avoid.


In [ ]:
Your response:


The next module writes binary data and object state to files. Those I/O streams move data between a program and a file. They serve a different purpose from the collection-processing streams used in this module.


## Supplemental Reading

- [Stream reduction contract](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/Stream.html#reduce(T,java.util.function.BinaryOperator)) defines identity and associative combination requirements.
- [Parallel pipelines and side effects](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/package-summary.html) explains stateless behavior, noninterference, and execution limits.
